In [14]:
import pandas as pandas


df = pandas.read_excel("/home/charen/corpus-corte-suprema-2026-08-15/scrap/salida/todo-corte-suprema-limpio.xlsx", engine="openpyxl")

In [15]:
df["Casacion/Apelacion"].value_counts()

Casacion/Apelacion
Casación                                113363
Recurso de Nulidad                       44763
Apelación                                34553
Revisión de Sentencia                     4456
Consulta                                  3695
Revisión                                  2423
Queja                                     1519
Extradición Activa                        1173
Casación Previsional                       831
Recurso de Queja Excepcional               571
Nulidad                                    520
Amparo                                     488
Queja NCPP                                  70
Acción Popular                              60
Competencia                                 60
Recurso de Queja Directa                    33
Medida Cautelar de Proceso de Amparo        24
Recurso de Queja Ordinaria                  12
Inhibición                                  11
Recusación                                   8
Extradición Pasiva                       

In [16]:
df["Link Resolucion"].is_unique
vc = df["Link Resolucion"].value_counts()
vc[vc > 1]
url = "https://jurisprudencia.pj.gob.pe/jurisprudenciaweb/ServletDescarga?uuid=c4c9df1c-de1b-4d0a-b33d-9e92b9ab1f37"
df[df["Link Resolucion"] == url]


,Casacion/Apelacion,Nro Expediente,Pretension/Delito,Tipo de Resolucion,Fecha de Resolucion,Sala Suprema,Norma de Derecho Interno (Articulo),Sumilla,Palabras Clave,Link Resolucion,...,Fecha,Anio,Anio Expediente,sumilla_tenia_prefijo,sumilla_util,fecha_incoherente,uuid_repetido,grupo_duplicado,es_duplicado_contenido,copia_descartable
13042,Apelación,000940-2008,NaN,Ejecutoria Suprema,17/07/2008,Sala de Derecho Constitucional y Social Perman...,NaN,NaN,NaN,https://jurisprudencia.pj.gob.pe/jurisprudenci...,...,2008-07-17,2008,2008,False,False,False,True,NaN,False,False
187452,Recurso de Nulidad,002562-2013,Promoción o Favorecimiento al Tráfico Ilícito ...,Ejecutoria Suprema,23/09/2014,Sala Penal Transitoria,Código Penal,NaN,Pena de multa,https://jurisprudencia.pj.gob.pe/jurisprudenci...,...,2014-09-23,2014,2013,False,False,False,True,NaN,False,False


In [17]:
sub = df[df.duplicated(subset=["Link Resolucion"], keep=False)]
sub["Link Resolucion"].tolist()

['https://jurisprudencia.pj.gob.pe/jurisprudenciaweb/ServletDescarga?uuid=c4c9df1c-de1b-4d0a-b33d-9e92b9ab1f37',
 'https://jurisprudencia.pj.gob.pe/jurisprudenciaweb/ServletDescarga?uuid=c4c9df1c-de1b-4d0a-b33d-9e92b9ab1f37']

In [18]:
df["Especialidad"].value_counts()

Especialidad
Penal                                 70912
Laboral                               31886
Contencioso Adm. Laboral              24724
Contencioso Administrativo            24114
Contencioso Adm. Previsional          14528
Civil                                 14198
Revision de Procedimiento Coactivo    13549
Constitucional                        11937
Familia Civil                          1834
Comercial                               517
Familia Penal                           270
Familia Tutelar                         172
Name: count, dtype: int64

In [ ]:
# Filtra "Especialidad" = Comercial y agrupa por año
comercial = df.loc[df["Especialidad"].astype(str).str.contains("comercial", case=False, na=False)].copy()

date_cols = [
    "Fecha de Resolucion",
    "Fecha Resolucion",
    "Fecha de la Resolucion",
    "Fecha de Publicacion",
    "Fecha de Publicación",
    "Fecha",
]

date_col = next((col for col in date_cols if col in comercial.columns), None)

if date_col is None:
    raise ValueError("No encontré una columna de fecha para agrupar por año. Revisa los nombres de columnas.")

comercial[date_col] = pandas.to_datetime(comercial[date_col], errors="coerce")
comercial["Anio"] = comercial[date_col].dt.year

resoluciones_comercial_por_anio = (
    comercial.groupby("Anio", dropna=False)
    .size()
    .rename("cantidad_resoluciones")
    .sort_index()
    .reset_index()
    .rename(columns={"Anio": "anio"})
)

resoluciones_comercial_por_anio